In [ ]:
import sys
sys.path.append('..')

from utils import DataHandler, Metric, train_model, set_seed, find_best_result_from_results_list
from models import MF

DEVICE = 'cuda'
INTERACTION_DATA_LIST = ['games']#['games', 'books', 'toys']
EMBEDDING_DIM_LIST = [8, 16, 32, 64, 128, 256]
NUM_EPOCHS = 200
PRIMARY_METRIC = 'NDCG@20'
SEED = 42


In [2]:
def run_single_experiment(
    interaction_data: str,
    embedding_dim: int,
    device: str = DEVICE,
) -> dict[str, float]:
    set_seed(SEED)

    datahandler = DataHandler(
        interaction_data=interaction_data,
        semantic_data=None,
        device=device,
        batch_size=4096,
        num_neg_item=64,
        seed=SEED,
    )
    metric = Metric(device=device)

    model = MF(
        rate_matrix=datahandler.rate_matrix,
        model_config=MF.ModelConfig(
            embedding_dim=embedding_dim,
            device=device,
            similarity='dot',
        ),
        loss_config=MF.InfoNCELossConfig(
            type='infonce',
            similarity='cos',
            temperature=0.3,
            is_inbatch=True,
            num_neg_item=64,
        ),
    )

    test_result = train_model(
        datahandler=datahandler,
        metric=metric,
        model=model,
        num_epochs=NUM_EPOCHS,
        primary_metric=PRIMARY_METRIC,
        verbose=False,
        use_amp=False,
    )
    return test_result


def grid_search_mf_embedding_dim(
    interaction_data: str,
    embedding_dim_list: list[int] = EMBEDDING_DIM_LIST,
) -> tuple[int, dict[str, float], list[dict[str, float]]]:
    print(f'\n=== MF Grid Search | dataset={interaction_data} ===')
    results = []

    for embedding_dim in embedding_dim_list:
        print(f'\n--- embedding_dim={embedding_dim} ---')
        result = run_single_experiment(
            interaction_data=interaction_data,
            embedding_dim=embedding_dim,
        )
        results.append(result)
        metrics_str = ', '.join(f'{k}: {v:.4f}' for k, v in result.items())
        print(f'embedding_dim={embedding_dim} | {metrics_str}')

    best_index, best_result = find_best_result_from_results_list(
        results_list=results,
        metric_name=PRIMARY_METRIC,
    )
    best_embedding_dim = embedding_dim_list[best_index]
    print(f'\nBest for {interaction_data}: embedding_dim={best_embedding_dim} | {best_result}')
    return best_embedding_dim, best_result, results


In [3]:
summary = {}

for interaction_data in INTERACTION_DATA_LIST:
    best_embedding_dim, best_result, _ = grid_search_mf_embedding_dim(interaction_data)
    summary[interaction_data] = {
        'best_embedding_dim': best_embedding_dim,
        'best_result': best_result,
    }

print('\n=== Summary ===')
for interaction_data, result in summary.items():
    print(
        f"{interaction_data}: best_embedding_dim={result['best_embedding_dim']} | {result['best_result']}"
    )

summary



=== MF Grid Search | dataset=games ===

--- embedding_dim=8 ---
Initial Model Performance (Before Training):
  Recall@10: 0.0037, NDCG@10: 0.0017, Recall@20: 0.0071, NDCG@20: 0.0025
Epoch [  100/2000]  Loss: 6.9930  Time: 19.4s   Recall@10: 0.0698, NDCG@10: 0.0350, Recall@20: 0.1142, NDCG@20: 0.0468


KeyboardInterrupt: 